<a href="https://colab.research.google.com/github/motahareh-ehsani/Position-Information-about-Fish/blob/main/improvedyolo_sharedmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import cv2
import os
import glob
import shutil
import re
from google.colab import drive
from ultralytics import YOLO

# ==================== SETUP ====================
drive.mount('/content/drive')
print("✓ Google Drive mounted")

# ==================== LOAD MODEL ====================
print("🤖 LOADING YOLO MODEL...")
try:
    model = YOLO('/content/fish_model.pt')
    print("✅ Model loaded successfully!")
    print(f"Model classes: {model.names}")
except Exception as e:
    print(f"❌ Failed to load model: {e}")
    print("💡 Make sure to upload your model file or use the correct path")
    exit()

# ==================== CREATE OUTPUT FOLDER ====================
print("📁 CREATING OUTPUT FOLDER ON DRIVE...")
output_backup_path = '/content/drive/MyDrive/colab_project/output_files'
os.makedirs(output_backup_path, exist_ok=True)
print(f"✅ Output folder created: {output_backup_path}")

# ==================== FILENAME FORMAT FILTER ====================
# Only process files matching: IP__date__starttime-endtime_IP
# Example: 192.168.20.101__2026-04-30__11-47-54-11-57-54_192.168.20.101
VALID_FILENAME_PATTERN = re.compile(
    r'^\d{1,3}(\.\d{1,3}){3}'   # IP address (e.g. 192.168.20.101)
    r'__'                         # double underscore
    r'\d{4}-\d{2}-\d{2}'         # date (e.g. 2026-04-30)
    r'__'                         # double underscore
    r'\d{2}-\d{2}-\d{2}'         # start time (e.g. 11-47-54)
    r'-'                          # dash separator
    r'\d{2}-\d{2}-\d{2}'         # end time (e.g. 11-57-54)
    r'_'                          # single underscore
    r'\d{1,3}(\.\d{1,3}){3}'     # IP address again
    r'(\.\w+)?$'                  # optional extension
)

def is_valid_camera_filename(filename):
    name_without_ext = os.path.splitext(filename)[0]
    return bool(VALID_FILENAME_PATTERN.match(name_without_ext))

# ==================== SMART VIDEO DISCOVERY (NO DUPLICATES) ====================
print("🔍 SMART VIDEO DISCOVERY (NO DUPLICATES)...")

input_backup_path = '/content/drive/MyDrive/colab_project/input_files'
os.makedirs(input_backup_path, exist_ok=True)

search_locations = ['/content']
video_extensions = ['.mp4', '.MP4', '.avi', '.mov', '.MOV', '.mkv', '.webm']
all_video_paths = []
found_filenames = set()
backed_up_count = 0
already_backed_up_count = 0
skipped_format_count = 0

print("📂 Searching for unique video files matching camera format...")
for location in search_locations:
    if os.path.exists(location):
        location_count = 0
        for root, dirs, files in os.walk(location):
            for file in files:
                if any(file.lower().endswith(ext.lower()) for ext in video_extensions):

                    # ✅ FORMAT FILTER: skip files that don't match the camera naming pattern
                    if not is_valid_camera_filename(file):
                        skipped_format_count += 1
                        print(f"   ⏭️  Skipped (wrong format): {file}")
                        continue

                    if file not in found_filenames:
                        full_path = os.path.join(root, file)
                        all_video_paths.append(full_path)
                        found_filenames.add(file)
                        location_count += 1

                        backup_dest = os.path.join(input_backup_path, file)
                        if not os.path.exists(backup_dest):
                            try:
                                shutil.copy2(full_path, backup_dest)
                                backed_up_count += 1
                                print(f"   📦 Backed up: {file}")
                            except Exception as e:
                                print(f"❌ Failed to backup {file}: {e}")
                        else:
                            already_backed_up_count += 1

        if location_count > 0:
            print(f"✅ Found {location_count} valid camera videos in {location}")

print(f"\n📊 BACKUP SUMMARY:")
print(f"📦 Newly backed up: {backed_up_count} files")
print(f"⏩ Already backed up: {already_backed_up_count} files")
print(f"⏭️  Skipped (wrong format): {skipped_format_count} files")
print(f"💾 Total in backup: {backed_up_count + already_backed_up_count} files")
print(f"✅ Input backup complete: {input_backup_path}")

all_video_paths = sorted(all_video_paths)
print(f"\n🎯 UNIQUE VALID VIDEOS FOUND: {len(all_video_paths)}")

print(f"\n📹 ALL VIDEO FILES ({len(all_video_paths)} total):")
for i, path in enumerate(all_video_paths):
    print(f"  {i+1:3d}. {os.path.basename(path)}")

# ==================== ANALYZE WHY VIDEOS AREN'T PROCESSING ====================
print(f"\n🔍 ANALYZING PROCESSING STATUS...")

results_dir = "/content/results"
os.makedirs(results_dir, exist_ok=True)

existing_local_results = []
if os.path.exists(results_dir):
    existing_local_results = [f for f in os.listdir(results_dir) if f.endswith('_fish_detailed_sizes.txt')]

existing_backup_results = []
if os.path.exists(output_backup_path):
    existing_backup_results = [f for f in os.listdir(output_backup_path) if f.endswith('_fish_detailed_sizes.txt')]

all_existing_results = set(existing_local_results + existing_backup_results)
print(f"📄 Existing result files: {len(all_existing_results)}")

videos_without_results = []
for video_path in all_video_paths:
    video_filename = os.path.basename(video_path)
    video_basename = os.path.splitext(video_filename)[0]
    expected_result_file = f"{video_basename}_fish_detailed_sizes.txt"

    if expected_result_file not in all_existing_results:
        videos_without_results.append(video_path)

print(f"\n📊 PROCESSING ANALYSIS:")
print(f"📹 Unique videos found: {len(all_video_paths)}")
print(f"✅ Videos with results: {len(all_existing_results)}")
print(f"🔄 Videos needing processing: {len(videos_without_results)}")

if videos_without_results:
    print(f"\n📝 VIDEOS THAT NEED PROCESSING:")
    for i, video in enumerate(videos_without_results[:20]):
        print(f"  {i+1:2d}. {os.path.basename(video)}")
    if len(videos_without_results) > 20:
        print(f"  ... and {len(videos_without_results) - 20} more")

# ==================== CHECK FOR COMMON ISSUES ====================
print(f"\n🔧 CHECKING FOR COMMON ISSUES...")
print(f"🔍 Validating videos that need processing...")

def is_video_valid(video_path):
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return False
        ret, frame = cap.read()
        cap.release()
        return ret and frame is not None
    except:
        return False

valid_videos_to_process = []
invalid_videos = []

for video_path in videos_without_results:
    if is_video_valid(video_path):
        valid_videos_to_process.append(video_path)
    else:
        invalid_videos.append(video_path)

print(f"📊 VALIDATION RESULTS:")
print(f"✅ Valid videos: {len(valid_videos_to_process)}")
print(f"❌ Invalid videos: {len(invalid_videos)}")

if invalid_videos:
    print(f"\n⚠️  Invalid videos (will be skipped):")
    for video in invalid_videos[:10]:
        print(f"   {os.path.basename(video)}")

# ==================== PROCESSING FUNCTION ====================
def process_video_with_detailed_info(video_path, output_file):
    try:
        print(f"🔍 Analyzing: {os.path.basename(video_path)}")

        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()

        results = model.predict(video_path, stream=True, conf=0.5)

        with open(output_file, 'w') as f:
            f.write("frame,center_x,center_y,width_px,height_px,fish_detected,confidence\n")

            detection_count = 0
            processed_frames = set()

            for frame_idx, r in enumerate(results):
                processed_frames.add(frame_idx)
                boxes = r.boxes

                if boxes is not None and len(boxes) > 0:
                    best_box = max(boxes, key=lambda b: float(b.conf[0]))

                    x1, y1, x2, y2 = best_box.xyxy[0].cpu().numpy()
                    width_px  = x2 - x1
                    height_px = y2 - y1
                    center_x  = (x1 + x2) / 2
                    center_y  = (y1 + y2) / 2
                    conf      = float(best_box.conf[0])

                    f.write(f"{frame_idx},{center_x:.2f},{center_y:.2f},"
                            f"{width_px:.2f},{height_px:.2f},yes,{conf:.4f}\n")
                    detection_count += 1

                else:
                    f.write(f"{frame_idx},,,,,no,0.0000\n")

            for frame_idx in range(total_frames):
                if frame_idx not in processed_frames:
                    f.write(f"{frame_idx},,,,,no,0.0000\n")

            no_detection_count = total_frames - detection_count
            f.write(f"\n")
            f.write(f"# ========== DETECTION SUMMARY ==========\n")
            f.write(f"# Total frames          : {total_frames}\n")
            f.write(f"# Frames with fish      : {detection_count} ({detection_count/total_frames*100:.1f}%)\n")
            f.write(f"# Frames without fish   : {no_detection_count} ({no_detection_count/total_frames*100:.1f}%)\n")
            f.write(f"# ========================================\n")

        print(f"✅ Complete: {detection_count}/{total_frames} frames with detections -> {output_file}")
        print(f"   🐟 Detected: {detection_count} ({detection_count/total_frames*100:.1f}%) | ❌ Missed: {no_detection_count} ({no_detection_count/total_frames*100:.1f}%)")

        backup_path = os.path.join(output_backup_path, os.path.basename(output_file))
        shutil.copy2(output_file, backup_path)
        print(f"📦 IMMEDIATELY BACKED UP TO DRIVE: {backup_path}")

        return True

    except Exception as e:
        print(f"❌ ERROR: {os.path.basename(video_path)} - {str(e)}")
        return False

# ==================== INITIALIZE COUNTERS ====================
successful_count = 0
failed_count = 0

# ==================== PROCESS THE VIDEOS ====================
if valid_videos_to_process:
    print(f"\n🎬 STARTING PROCESSING OF {len(valid_videos_to_process)} VIDEOS...")

    for i, video_path in enumerate(valid_videos_to_process, 1):
        print(f"\n{'='*60}")
        print(f"Processing [{i}/{len(valid_videos_to_process)}]: {os.path.basename(video_path)}")
        print(f"{'='*60}")

        base_name = os.path.splitext(os.path.basename(video_path))[0]
        output_file = os.path.join(results_dir, f"{base_name}_fish_detailed_sizes.txt")

        success = process_video_with_detailed_info(video_path, output_file)

        if success:
            successful_count += 1
        else:
            failed_count += 1

        if i % 5 == 0:
            print(f"\n📈 Progress: {i}/{len(valid_videos_to_process)}")
            print(f"   ✅ Successful: {successful_count} | ❌ Failed: {failed_count}")

    print(f"\n🎉 PROCESSING COMPLETED!")
    print(f"   ✅ Successful: {successful_count}")
    print(f"   ❌ Failed: {failed_count}")

else:
    print(f"\n💡 No valid videos need processing!")

# ==================== FINAL SUMMARY ====================
print(f"\n{'='*60}")
print("📊 FINAL SUMMARY")
print(f"{'='*60}")

total_all_results = len(all_existing_results) + successful_count

print(f"📹 Unique videos discovered: {len(all_video_paths)}")
print(f"📦 Total files in backup: {backed_up_count + already_backed_up_count}")
print(f"📄 Total result files: {total_all_results}")
completion_pct = (total_all_results / len(all_video_paths) * 100) if len(all_video_paths) > 0 else 0
print(f"📈 Processing completion: {total_all_results}/{len(all_video_paths)} ({completion_pct:.1f}%)")

if total_all_results >= len(all_video_paths):
    print("🎉 CONGRATULATIONS! All videos processed!")
elif total_all_results > 0:
    print(f"🔄 Remaining: {len(all_video_paths) - total_all_results} videos")
else:
    print("❌ No videos were processed")

print(f"\n📁 INPUT BACKUP: {input_backup_path}")
print(f"📁 OUTPUT BACKUP: {output_backup_path}")
print(f"🚀 Script completed!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted
🤖 LOADING YOLO MODEL...
❌ Failed to load model: [Errno 2] No such file or directory: '/content/fish_model.pt'
💡 Make sure to upload your model file or use the correct path
📁 CREATING OUTPUT FOLDER ON DRIVE...
✅ Output folder created: /content/drive/MyDrive/colab_project/output_files
🔍 SMART VIDEO DISCOVERY (NO DUPLICATES)...
📂 Searching for unique video files matching camera format...
   ⏭️  Skipped (wrong format): VID_20251012_203638.mp4
   ⏭️  Skipped (wrong format): VID_20251012_191826.mp4
   ⏭️  Skipped (wrong format): VID_20251012_191726.mp4
   ⏭️  Skipped (wrong format): fish_320x240.mp4
   ⏭️  Skipped (wrong format): fish_320x240.avi
   ⏭️  Skipped (wrong format): fish_320x240.avi
   ⏭️  Skipped (wrong format): 19-51-50-20-01-50.avi
   ⏭️  Skipped (wrong format): fish_320x240.mp4
   ⏭️  Skipped (wrong format): fish_320x240.avi
  